# IRC Labels: Governance via Trusted Engine

Catalog classifies -> Labels travel via IRC -> ClickHouse enforces natively.

**Prerequisites**: Run `setup/bootstrap.py` and `demo.ipynb` first.

In [54]:
import sys
sys.path.insert(0, '.')

from pyiceberg.catalog import load_catalog
import clickhouse_connect

S3 = {"s3.endpoint": "http://localhost:9000", "s3.access-key-id": "admin",
      "s3.secret-access-key": "password", "s3.region": "us-east-1"}

catalog = load_catalog("healthcare", type="rest",
    uri="http://localhost:8182/catalog", warehouse="healthcare", **S3)

ch = clickhouse_connect.get_client(host='localhost', port=8123, password='demo')

# Create ClickHouse database connected to Lakekeeper via IRC
try:
    ch.command("SET allow_database_iceberg = 1")
    ch.command("""
        CREATE DATABASE IF NOT EXISTS healthcare
        ENGINE = DataLakeCatalog('http://lakekeeper:8181/catalog')
        SETTINGS catalog_type = 'rest', warehouse = 'healthcare',
                 storage_endpoint = 'http://minio:9000/warehouse'
    """)
    print('ClickHouse database connected to Lakekeeper via IRC')
except Exception as e:
    print(f'DB setup: {e}')

print('Ready')


ClickHouse database connected to Lakekeeper via IRC
Ready


---
## Part 1: Populate `iceberg_labels` from IRC

Read labels from LoadTableResponse and make them queryable in ClickHouse SQL.

In [56]:
import sys; sys.path.insert(0, "agent")
from governance_engine import populate_iceberg_labels

count = populate_iceberg_labels(catalog, ch, "healthcare")
print(f"Loaded {count} labels into iceberg_labels\n")

result = ch.query("""
    SELECT database, table_name, scope, column_name, label_key, label_value
    FROM iceberg_labels ORDER BY table_name, scope DESC, label_key
""")
print(f"{'db':12s} {'table':18s} {'scope':8s} {'column':12s} {'key':20s} {'value'}")
print("-" * 90)

Loaded 59 labels into iceberg_labels

db           table              scope    column       key                  value
------------------------------------------------------------------------------------------


In [57]:
# Discovery via SQL
print("=== Sensitive tables ===")
for row in ch.query("SELECT table_name, label_value FROM iceberg_labels WHERE label_key='sensitivity' AND scope='table' ORDER BY label_value DESC").result_rows:
    print(f"  {row[0]:20s} sensitivity={row[1]}")

print("\n=== PII columns ===")
for row in ch.query("SELECT table_name, column_name, label_key, label_value FROM iceberg_labels WHERE label_key IN ('pii_type','phi_type')").result_rows:
    print(f"  {row[0]:18s} {row[1]:15s} {row[2]}={row[3]}")

=== Sensitive tables ===
  patients             sensitivity=restricted
  billing              sensitivity=medium
  visits_summary       sensitivity=low

=== PII columns ===
  patients           diagnosis       phi_type=clinical_diagnosis
  patients           name            pii_type=full_name
  patients           email           pii_type=email
  patients           dob             pii_type=date_of_birth


---
## Part 2: Label-Driven Governance

One function call: read labels -> generate governed views with masking and blocking.

In [33]:
import sys; sys.path.insert(0, "agent")
from governance_engine import apply_label_governance

views = apply_label_governance(catalog, ch, "healthcare", role="analyst")

for table_name, info in views.items():
    print(f"\n{'='*70}")
    print(f"Table: {table_name} -> View: {info['view']}")
    print(f"{'='*70}")
    print(info['ddl'])


Loaded 59 labels into iceberg_labels table


---
## Part 3: Raw vs Governed — Side by Side

In [39]:
# Side-by-side: raw vs governed
raw = ch.query("SELECT name, email, diagnosis FROM `healthcare`.`healthcare.patients` LIMIT 4").result_rows
gov = ch.query("SELECT name, email, diagnosis FROM default.`governed_patients` LIMIT 4").result_rows

print(f"{'RAW':42s} | GOVERNED")
print(f"{'name':14s} {'email':20s} {'diagnosis':10s} | {'name':14s} {'email':20s} {'diagnosis'}")
print("-" * 44 + "|" + "-" * 50)
for r, g in zip(raw, gov):
    print(f"{str(r[0]):14s} {str(r[1]):20s} {str(r[2]):10s} | {str(g[0]):14s} {str(g[1]):20s} {str(g[2])}")


DatabaseError: Received ClickHouse exception, code: 36, server response: Code: 36. DB::Exception: Table cannot have empty namespace: governed_patients. (BAD_ARGUMENTS) (for url http://localhost:8123)

In [38]:
# Safe table passes through
print("=== visits_summary (sensitivity=low) -> no masking ===")
result = ch.query("SELECT * FROM default.`governed_visits_summary` LIMIT 5")
for col in result.column_names:
    print(f"  {col:20s}", end="")
print()
for row in result.result_rows:
    for val in row:
        print(f"  {str(val):20s}", end="")
    print()


=== visits_summary (sensitivity=low) -> no masking ===
  visit_date            department            visit_count           avg_wait_time         satisfaction_score  
  2026-01-05            Cardiology            45                    22.5                  4.2                 
  2026-01-05            Neurology             32                    18.3                  4.5                 
  2026-01-05            General               128                   35.2                  3.8                 
  2026-01-12            Cardiology            52                    24.1                  4.1                 
  2026-01-12            Neurology             28                    17.8                  4.6                 


---
## Part 4: Compliance Inventory — Pure SQL

In [36]:
print("=== HIPAA Tables ===")
for (tbl,) in ch.query("SELECT table_name FROM iceberg_labels WHERE label_key='regulatory_scope' AND label_value LIKE '%HIPAA%'").result_rows:
    pii = ch.query(f"SELECT column_name, label_key, label_value FROM iceberg_labels WHERE table_name='{tbl}' AND label_key IN ('pii_type','phi_type')").result_rows
    print(f"\n  {tbl}: {len(pii)} PII/PHI columns")
    for col, lk, lv in pii:
        print(f"    {col:15s} {lk}={lv}")

print("\n=== Regulation Coverage ===")
for row in ch.query("SELECT label_value, groupArray(table_name), count() FROM iceberg_labels WHERE label_key='regulatory_scope' GROUP BY label_value").result_rows:
    print(f"  {row[0]:10s} {row[2]} table(s): {row[1]}")

=== HIPAA Tables ===

  patients: 4 PII/PHI columns
    diagnosis       phi_type=clinical_diagnosis
    name            pii_type=full_name
    email           pii_type=email
    dob             pii_type=date_of_birth

=== Regulation Coverage ===
  SOX        1 table(s): ['billing']
  HIPAA      1 table(s): ['patients']


---
## Part 5: Role-Based Access — Same View, Different Data

Create a ClickHouse user with a different role. Same governed view, different results.

In [ ]:
# Create an 'admin' user who sees unmasked data
try:
    ch.command("CREATE USER IF NOT EXISTS analyst IDENTIFIED BY 'analyst'")
    ch.command("GRANT SELECT ON default.* TO analyst")
    ch.command("GRANT SELECT ON healthcare.* TO analyst")
    print('Created user: analyst')
except Exception as e:
    print(f'User setup: {e}')

# The governed view masks based on currentUser()
# 'default' user -> masked, 'analyst' could have different masking
import clickhouse_connect

ch_admin = clickhouse_connect.get_client(host='localhost', port=8123, password='demo', username='default')
ch_analyst = clickhouse_connect.get_client(host='localhost', port=8123, password='analyst', username='analyst')

print('\n=== default user (admin) ===')
for row in ch_admin.query('SELECT name, email, diagnosis FROM default.`governed_patients` LIMIT 4').result_rows:
    print(f'  {row[0]:15s} {row[1]:25s} {str(row[2])}')

print('\n=== analyst user (masked) ===')
for row in ch_analyst.query('SELECT name, email, diagnosis FROM default.`governed_patients` LIMIT 4').result_rows:
    print(f'  {row[0]:15s} {row[1]:25s} {str(row[2])}')

print('\n-> Same view, same SQL. Labels + role = different results.')


---
## Part 6: Change a Label — Governance Updates Automatically

Change the sensitivity label on the `email` column from `high` to `low`.
The governed view immediately reflects the change — no DDL, no redeployment.

In [ ]:
# Query as analyst (who sees masked data)
ch_analyst = clickhouse_connect.get_client(host='localhost', port=8123, password='analyst', username='analyst')

print('=== BEFORE: analyst sees email masked ===')
for row in ch_analyst.query('SELECT name, email FROM default.`governed_patients` LIMIT 3').result_rows:
    print(f'  {row[0]:15s} {row[1]}')

# Change label: remove pii_type from email column, downgrade sensitivity
print('\n--- Changing label: remove pii_type, set sensitivity=low ---')

ch.command("ALTER TABLE iceberg_labels DELETE WHERE table_name='patients' AND column_name='email' AND label_key='pii_type'")
ch.command("ALTER TABLE iceberg_labels UPDATE label_value='low' WHERE table_name='patients' AND column_name='email' AND label_key='sensitivity'")

# Wait for mutations to complete
import time
for _ in range(10):
    pending = ch.query('SELECT count() FROM system.mutations WHERE is_done=0').result_rows[0][0]
    if pending == 0:
        break
    time.sleep(1)
print(f'  Mutations complete')

# Regenerate governed view from updated labels
from governance_engine import generate_governed_view
result = generate_governed_view(ch, 'healthcare', 'patients', role='analyst')
if result:
    print(f'  Regenerated view: {result[0]}')
else:
    print('  View regeneration returned None')

print('\n=== AFTER: analyst now sees email (sensitivity=low, no pii_type) ===')
# Reconnect to clear any caching
ch_analyst2 = clickhouse_connect.get_client(host='localhost', port=8123, password='analyst', username='analyst')
for row in ch_analyst2.query('SELECT name, email FROM default.`governed_patients` LIMIT 3').result_rows:
    print(f'  {row[0]:15s} {row[1]}')

print('\n-> Label changed. View regenerated. Email now visible for analyst.')
print('   In production: catalog label change -> engine picks up automatically.')


---
## Key Takeaway

``
Catalog classifies (labels)  ->  IRC carries intent  ->  Engine enforces natively
``

One classification. Every engine enforces. No runtime coordination.
